## Sales Data Cleaning

Read the first file to see the structure of the data.

In [0]:

sales17 = spark.table("accenture_final_project.bronze_layer.sales_2017")


Concatenate the data from all the years into one Dataframe

In [0]:
sales17 = spark.table("accenture_final_project.bronze_layer.sales_2017")
sales18 = spark.table("accenture_final_project.bronze_layer.sales_2018")
sales19 = spark.table("accenture_final_project.bronze_layer.sales_2019")
sales20 = spark.table("accenture_final_project.bronze_layer.sales_2020")

sales = sales17.unionByName(sales18)\
                .unionByName(sales19)\
                .unionByName(sales20)



Correcting the names of the columns to not have any trouble

In [0]:
sales = sales.withColumnRenamed("Unit_Price", "UnitPrice")

Making the order date column as date type from string.

In [0]:
from pyspark.sql.functions import regexp_replace, to_date, col , when , length

sales = sales.withColumn(
    "OrderDate",
    to_date(regexp_replace("OrderDate", r"^[A-Za-z]+,\s*", ""), "MMMM d, yyyy")
)

Removing the $ sign from currency columns and making them double type.

In [0]:
sales = sales.withColumn("UnitPrice",
        regexp_replace("UnitPrice", "[$,]", "").cast("double")
)

sales = sales.withColumn("Sales",
        regexp_replace("Sales", "[$,]", "").cast("double")
)

sales = sales.withColumn("Cost",
        regexp_replace("Cost", "[$,]", "").cast("double")
)

 We identified Product Keys that where 4-digits , we count them and we decided to erase the last digit as it was zero in all five examples. There were only 5 out of 400 , and we decided that it was a typo.

In [0]:
sales.filter(
    (col("ProductKey") >= 1000) & (col("ProductKey") <= 9999)
).show()

In [0]:
sales = sales.withColumn(
    "ProductKey",
    when(
        length(col("ProductKey").cast("string")) == 4,
        regexp_replace(col("ProductKey").cast("string"), "0$", "").cast("int")
    ).otherwise(col("ProductKey"))
)

Making the other string type columns integer type.

In [0]:
sales = sales.withColumn("ProductKey", col("ProductKey").cast("int")) \
       .withColumn("ResellerKey", col("ResellerKey").cast("int")) \
       .withColumn("EmployeeKey", col("EmployeeKey").cast("int")) \
       .withColumn("SalesTerritoryKey", col("SalesTerritoryKey").cast("int")) \
       .withColumn("Quantity", col("Quantity").cast("int"))

Seeing the correct schema

In [0]:
sales.printSchema()

Check for Null Values

In [0]:
from pyspark.sql.functions import col, sum

sales.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in sales.columns
]).show()

Check fo duplicate rows

In [0]:
total = sales.count()
unique = sales.distinct().count()

print("Total rows:", total)
print("Unique rows:", unique)
print("Duplicate rows:", total - unique)

### Checking for Outliers and Bad data.

Check for negative values

In [0]:
sales.filter(
    (col("Quantity") < 0) |
    (col("UnitPrice") < 0) |
    (col("Sales") < 0) |
    (col("Cost") < 0)
    ).show()

Basic statistics for our data.

In [0]:
sales.describe().show()

Check for outliers with Interquartile Range (IQR) Method.

In [0]:
from pyspark.sql.functions import col

# calculate quartiles
q1, q3 = sales.approxQuantile("Sales", [0.25, 0.75], 0)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

outliers = sales.filter(
    (col("Sales") < lower_bound) | (col("Sales") > upper_bound)
)

outliers.show()

In [0]:
outliers.count()

Boxplot to see also about outliers!

In [0]:
import matplotlib.pyplot as plt

# convert only the needed column to pandas
sales_pd = sales.select("Sales").toPandas()

plt.boxplot(sales_pd["Sales"])
plt.title("Boxplot of Sales")
plt.ylabel("Sales")
plt.show()

The numeric columns were inspected for extreme values using summary statistics.
Large values were observed in the Sales column; however, these correspond to high-value product transactions and bulk orders. Since the AdventureWorks dataset represents realistic sales activity, these values were retained.

The sales data are clean!

### Creating the delta table 

In [0]:
sales_silver = sales

sales_silver.write \
    .format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("accenture_final_project.silver_layer.sales")

In [0]:
display(spark.table("accenture_final_project.silver_layer.sales"))